In [1]:
SPOT_PATH = '../data/trb_usdt_spot_export.csv'
PERP_PATH = '../data/trb_usdt_futures_export.csv'
TRADES_PATH = '../data/trb_usdt_trades_export.csv'

In [2]:
import logging
from logging.handlers import RotatingFileHandler
import sys
import pandas as pd
from datetime import datetime

def setup_logging(log_file='pipeline.log'):
    """Configure logging for the pipeline"""
    logger = logging.getLogger('market_analysis')
    logger.setLevel(logging.DEBUG)
    
    # Clear existing handlers
    logger.handlers = []
    
    # Format for our log messages
    formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Console handler (shows INFO and above)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)
    
    # File handler (logs everything DEBUG and above)
    file_handler = RotatingFileHandler(
        log_file,
        maxBytes=10*1024*1024,  # 10MB
        backupCount=5
    )
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)
    
    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    
    return logger

logger = setup_logging()

# 1. Data Preparation and Synchronization


In [27]:
import re
import pandas as pd
import numpy as np
import logging
from typing import Optional, Tuple, Union

### ----------------------------
### 1. TIMESTAMP PARSING UTILITY
### ----------------------------

def parse_timestamps_robustly(
    series: Union[pd.Series, list], 
    debug_name: str = "data"
) -> pd.Series:
    """
    Robust timestamp parser that handles multiple datetime formats.
    """
    logger = logging.getLogger(f'market_analysis.timestamp_parser.{debug_name}')
    
    if not isinstance(series, pd.Series):
        series = pd.Series(series)
    
    if not isinstance(series.iloc[0], str):
        series = series.astype(str)

    no_tz = series.str.replace(r'[\+\-]\d{2}:?\d{0,2}$', '', regex=True)
    
    try:
        parsed = pd.to_datetime(no_tz, format='ISO8601', utc=True)
    except ValueError:
        try:
            parsed = pd.to_datetime(no_tz, format='mixed', utc=True)
        except Exception as e:
            logger.error(f"Failed to parse timestamps: {str(e)}")
            raise ValueError(f"Could not parse {debug_name} timestamps") from e
    
    if parsed.isna().any():
        raise ValueError(f"Failed to parse some {debug_name} timestamps")
    
    return parsed

### ----------------------------
### 2. PREPROCESSING PIPELINE
### ----------------------------

def preprocess_data(
    spot_data: pd.DataFrame,
    perp_data: pd.DataFrame,
    trade_data: Optional[pd.DataFrame] = None,
    strict_validation: bool = True
) -> Tuple[pd.DataFrame, pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Fixed preprocessing pipeline that handles DataFrame comparisons correctly.
    """
    logger = logging.getLogger('market_analysis.preprocessor')
    
    try:
        # Standardize column names
        def standardize_cols(df: pd.DataFrame, suffix: str) -> pd.DataFrame:
            if df is None:
                return None
                
            cols = {
                'time': 'timestamp',
                'bid_price': f'best_bid_{suffix}',
                'ask_price': f'best_ask_{suffix}',
                'price': 'trade_price',
                'quantity': 'trade_quantity'
            }
            return df.rename(columns={k: v for k, v in cols.items() if k in df.columns})

        spot_data = standardize_cols(spot_data, 'spot')
        perp_data = standardize_cols(perp_data, 'perp')
        trade_data = standardize_cols(trade_data, 'trade') if trade_data is not None else None

        # Parse timestamps
        for df, name in [(spot_data, 'spot'), (perp_data, 'perp'), (trade_data, 'trade')]:
            if df is not None:
                df['timestamp'] = parse_timestamps_robustly(df['timestamp'], name)
                df.sort_values('timestamp', inplace=True)

        # Convert numeric columns
        def convert_numeric(df: pd.DataFrame, cols: list) -> None:
            if df is not None:
                for col in cols:
                    if col in df.columns:
                        df[col] = pd.to_numeric(df[col], errors='coerce').ffill()

        convert_numeric(spot_data, ['best_bid_spot', 'best_ask_spot'])
        convert_numeric(perp_data, ['best_bid_perp', 'best_ask_perp'])
        if trade_data is not None:
            convert_numeric(trade_data, ['trade_price', 'trade_quantity'])

        # Time alignment
        def get_time_bounds(df):
            return (df['timestamp'].min(), df['timestamp'].max()) if df is not None else (None, None)

        spot_min, spot_max = get_time_bounds(spot_data)
        perp_min, perp_max = get_time_bounds(perp_data)
        
        min_time = max(t for t in [spot_min, perp_min] if t is not None)
        max_time = min(t for t in [spot_max, perp_max] if t is not None)

        # Filter to common time period
        def filter_time(df, min_t, max_t):
            if df is not None:
                mask = (df['timestamp'] >= min_t) & (df['timestamp'] <= max_t)
                return df.loc[mask].copy()
            return None

        spot_data = filter_time(spot_data, min_time, max_time)
        perp_data = filter_time(perp_data, min_time, max_time)
        trade_data = filter_time(trade_data, min_time, max_time) if trade_data is not None else None

        # Drop remaining NAs in critical columns
        spot_data = spot_data.dropna(subset=['best_bid_spot', 'best_ask_spot'])
        perp_data = perp_data.dropna(subset=['best_bid_perp', 'best_ask_perp'])
        if trade_data is not None:
            trade_data = trade_data.dropna(subset=['trade_price'])

        logger.info(
            f"Preprocessing complete\n"
            f"- Spot: {len(spot_data)} rows\n"
            f"- Perp: {len(perp_data)} rows\n"
            f"- Trade: {len(trade_data) if trade_data is not None else 0} rows"
        )

        return spot_data, perp_data, trade_data

    except Exception as e:
        logger.error(f"Preprocessing failed: {str(e)}")
        raise ValueError("Preprocessing error") from e

# 2. Primary Objective: Predictive Signal Generation


## 2.1 Sudden Price Movement Detection


In [ ]:
### ----------------------------
### 3. ANALYSIS FUNCTIONS
### ----------------------------

def detect_price_jumps(data, lookback_ms=3, threshold=0.0007):
    """Detect significant price movements"""
    data = data.copy().sort_values('timestamp')
    
    # Calculate rolling changes using fixed window sizes
    data['spot_pct_change'] = data['mid_spot'].pct_change().rolling(window=lookback_ms).sum()
    data['perp_pct_change'] = data['mid_perp'].pct_change().rolling(window=lookback_ms).sum()
    
    # Flag significant movements
    data['spot_jump'] = (abs(data['spot_pct_change']) >= threshold).astype(int)
    data['perp_jump'] = (abs(data['perp_pct_change']) >= threshold).astype(int)
    
    # Add direction information
    data['spot_direction'] = np.sign(data['spot_pct_change'])
    data['perp_direction'] = np.sign(data['perp_pct_change'])
    
    return data

## 2.2 Cross-Market Impact Analysis


In [ ]:
def analyze_market_impact(jump_data, window_ms=5):
    """Analyze cross-market impact of price jumps"""
    results = []
    jump_data = jump_data.sort_values('timestamp')
    
    # Spot to Perpetual impact
    spot_jumps = jump_data[jump_data['spot_jump'] == 1]
    for _, row in spot_jumps.iterrows():
        future_window = jump_data[
            (jump_data['timestamp'] > row['timestamp']) & 
            (jump_data['timestamp'] <= row['timestamp'] + timedelta(milliseconds=window_ms))
        ]
        if not future_window.empty:
            impact = (future_window['mid_perp'].iloc[-1] - row['mid_perp']) / row['mid_perp']
            results.append({
                'type': 'spot_to_perp',
                'timestamp': row['timestamp'],
                'initiator_direction': row['spot_direction'],
                'impact': impact,
                'initiator_magnitude': row['spot_pct_change'],
                'response_magnitude': impact
            })
    
    # Perpetual to Spot impact
    perp_jumps = jump_data[jump_data['perp_jump'] == 1]
    for _, row in perp_jumps.iterrows():
        future_window = jump_data[
            (jump_data['timestamp'] > row['timestamp']) & 
            (jump_data['timestamp'] <= row['timestamp'] + timedelta(milliseconds=window_ms))
        ]
        if not future_window.empty:
            impact = (future_window['mid_spot'].iloc[-1] - row['mid_spot']) / row['mid_spot']
            results.append({
                'type': 'perp_to_spot',
                'timestamp': row['timestamp'],
                'initiator_direction': row['perp_direction'],
                'impact': impact,
                'initiator_magnitude': row['perp_pct_change'],
                'response_magnitude': impact
            })
    
    return pd.DataFrame(results)

## 2.3 Predictive Model Implementation


In [25]:
from typing import Any, Optional, Tuple
import pandas as pd
import numpy as np
import logging
from sklearn.ensemble import GradientBoostingRegressor

### ----------------------------
### 2. ENHANCED FEATURE ENGINEERING (FIXED)
### ----------------------------
def create_features(spot_data: pd.DataFrame, 
                   perp_data: pd.DataFrame, 
                   trade_data: Optional[pd.DataFrame] = None,
                   time_tolerance: str = '10ms') -> pd.DataFrame:
    """
    Ultimate robust feature engineering pipeline with:
    - Comprehensive data validation
    - Graceful handling of missing values
    - Detailed diagnostics
    - Maximum data preservation
    """
    logger = logging.getLogger('market_analysis.feature_engineering')
    logger.info("Creating features...")
    
    try:
        # 1. Initial data validation
        logger.info("Validating input data...")
        def validate_data(df, name):
            if df is None or len(df) == 0:
                raise ValueError(f"{name} data is empty or None")
            if 'timestamp' not in df.columns:
                raise ValueError(f"{name} data missing timestamp column")
            return df.sort_values('timestamp')

        spot_data = validate_data(spot_data, "Spot")
        perp_data = validate_data(perp_data, "Perpetual")
        if trade_data is not None:
            trade_data = validate_data(trade_data, "Trade")

        # 2. Initial merge with diagnostics
        logger.info("Merging spot and perpetual data...")
        try:
            features = pd.merge_asof(
                spot_data,
                perp_data,
                on='timestamp',
                direction='nearest',
                tolerance=pd.Timedelta(time_tolerance),
                suffixes=('', '_perp')
            )
            logger.info(f"Merged data contains {len(features)} initial rows")
        except Exception as e:
            logger.error("Merge failed. Sample data:")
            logger.error(f"Spot head:\n{spot_data.head()}")
            logger.error(f"Perp head:\n{perp_data.head()}")
            raise ValueError(f"Merge failed: {str(e)}")

        # 3. Calculate market features with protection
        logger.info("Calculating market features...")
        required_market_cols = ['best_bid_spot', 'best_ask_spot', 'best_bid_perp', 'best_ask_perp']
        missing_market_cols = [col for col in required_market_cols if col not in features.columns]
        
        if missing_market_cols:
            logger.error(f"Missing market columns: {missing_market_cols}")
            logger.info("Available columns: " + ", ".join(features.columns))
            raise ValueError(f"Missing required market columns: {missing_market_cols}")

        # Safe calculation with NaN protection
        with np.errstate(all='ignore'):
            features['mid_spot'] = (features['best_bid_spot'] + features['best_ask_spot']) / 2
            features['mid_perp'] = (features['best_bid_perp'] + features['best_ask_perp']) / 2
            features['premium'] = features['mid_perp'] - features['mid_spot']
            features['premium_pct'] = np.where(
                features['mid_spot'] != 0,
                features['premium'] / features['mid_spot'],
                np.nan
            )
            features['spot_spread'] = features['best_ask_spot'] - features['best_bid_spot']
            features['perp_spread'] = features['best_ask_perp'] - features['best_bid_perp']

        # 4. Trade data incorporation with robust handling
        if trade_data is not None:
            logger.info("Processing trade data...")
            try:
                # Check for required trade columns
                trade_cols = ['trade_price', 'trade_quantity']
                available_trade_cols = [col for col in trade_cols if col in trade_data.columns]
                
                if not available_trade_cols:
                    logger.warning("No valid trade data columns available")
                else:
                    # Handle duplicates and aggregate
                    trade_data = trade_data.drop_duplicates(subset='timestamp', keep='last')
                    
                    agg_dict = {
                        'trade_price': ['mean', 'std', 'last'],
                        'trade_quantity': 'sum'
                    }
                    # Only use available columns
                    agg_dict = {k: v for k, v in agg_dict.items() if k in trade_data.columns}
                    
                    trade_features = trade_data.groupby('timestamp').agg(agg_dict)
                    trade_features.columns = ['trade_' + '_'.join(col).strip() 
                                           for col in trade_features.columns.values]
                    
                    # Merge with tolerance for imperfect alignment
                    features = pd.merge_asof(
                        features.sort_values('timestamp'),
                        trade_features.sort_values('timestamp'),
                        on='timestamp',
                        direction='nearest',
                        tolerance=pd.Timedelta(time_tolerance)
                    )
                    logger.info(f"Added {len(trade_features.columns)} trade features")
            except Exception as e:
                logger.warning(f"Trade data processing failed: {str(e)}")

        # 5. Advanced NaN handling with diagnostics
        logger.info("Handling missing values with diagnostics...")
        
        # Pre-cleaning diagnostics
        logger.info(f"Pre-cleaning: {len(features)} rows, {features.isna().sum().sum()} NaNs")
        
        # Critical market data - must have these
        critical_cols = ['mid_spot', 'mid_perp']
        pre_critical_count = len(features)
        features = features.dropna(subset=critical_cols)
        post_critical_count = len(features)
        
        if post_critical_count == 0:
            logger.error("All rows dropped after critical columns check")
            logger.error(f"NaN counts in critical columns:\n{features[critical_cols].isna().sum()}")
            raise ValueError("No rows with complete market data")
        
        logger.info(f"Retained {post_critical_count}/{pre_critical_count} rows after critical check")

        # Trade data - fill NaNs intelligently
        trade_cols = [col for col in features.columns if col.startswith('trade_')]
        for col in trade_cols:
            if col in features.columns and features[col].isna().any():
                if col.endswith('_std'):
                    fill_val = features[col].mean()
                    logger.info(f"Filling {col} NaNs with mean: {fill_val}")
                else:
                    fill_val = 0
                    logger.info(f"Filling {col} NaNs with 0")
                features[col] = features[col].fillna(fill_val)

        # Final validation
        remaining_nans = features.columns[features.isna().any()].tolist()
        if remaining_nans:
            logger.warning(f"Final NaN columns: {remaining_nans}")
            logger.warning(f"NaN counts:\n{features[remaining_nans].isna().sum()}")
            
            # Only drop if absolutely necessary
            features = features.dropna(subset=remaining_nans)
            logger.info(f"Final row count after NaN drop: {len(features)}")

        if len(features) == 0:
            raise ValueError("No valid rows - check input data quality")
        
        logger.info(f"Feature engineering complete. Final shape: {features.shape}")
        return features

    except Exception as e:
        logger.error("Feature engineering failed with details:")
        if 'features' in locals():
            logger.error(f"Data shape at failure: {features.shape if hasattr(features, 'shape') else 'No features'}")
            logger.error(f"NaN summary:\n{features.isna().sum() if hasattr(features, 'isna') else 'No features'}")
        logger.error(f"Error: {str(e)}")
        raise ValueError(f"Feature engineering failed: {str(e)}") from e
### ----------------------------
### 4. ROBUST MODEL BUILDING (UNCHANGED)
### ----------------------------

def build_predictive_models(features: pd.DataFrame,
                          test_size: float = 0.2,
                          min_samples: int = 10000) -> Tuple[Any, Any]:
    """
    Enhanced model building with:
    - Temporal cross-validation
    - Comprehensive feature validation
    - NaN handling
    - Model evaluation
    
    Args:
        features: Engineered features DataFrame
        test_size: Proportion of data for testing
        min_samples: Minimum required samples
        
    Returns:
        Tuple of (spot_model, perp_model)
    """
    logger = logging.getLogger('market_analysis.modeling')
    logger.info("Building models...")
    
    try:
        features = features.copy().sort_values('timestamp')
        
        # 1. Create targets with lookahead
        lookahead = 5  # Number of periods to predict ahead
        features['spot_target'] = features['mid_spot'].shift(-lookahead)
        features['perp_target'] = features['mid_perp'].shift(-lookahead)
        
        # 2. Validate and clean data
        initial_count = len(features)
        features = features.dropna(subset=['spot_target', 'perp_target'])
        
        if len(features) < min_samples:
            raise ValueError(
                f"Insufficient data: {len(features)} samples after cleaning "
                f"(need at least {min_samples}). Initial: {initial_count}"
            )
        
        # 3. Select and validate features
        numeric_cols = features.select_dtypes(include=[np.number]).columns
        exclude_cols = {'timestamp', 'spot_target', 'perp_target', 'mid_spot', 'mid_perp'}
        feature_cols = [col for col in numeric_cols if col not in exclude_cols]
        
        X = features[feature_cols]
        y_spot = features['spot_target']
        y_perp = features['perp_target']
        
        # Final NaN check
        if X.isna().any().any():
            nan_cols = X.columns[X.isna().any()].tolist()
            raise ValueError(
                f"Modeling failed - NaN values in features: {nan_cols}\n"
                f"Preprocessing steps should eliminate all NaNs. "
                f"Check your feature engineering pipeline."
            )
        
        # 4. Temporal train-test split
        split_idx = int(len(X) * (1 - test_size))
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_spot_train, y_spot_test = y_spot.iloc[:split_idx], y_spot.iloc[split_idx:]
        y_perp_train, y_perp_test = y_perp.iloc[:split_idx], y_perp.iloc[split_idx:]
        
        logger.info(
            f"Train/test split: {len(X_train)}/{len(X_test)} samples "
            f"(features: {len(feature_cols)})"
        )
        
        # 5. Model training with validation
        spot_model = GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            random_state=42
        )
        
        perp_model = GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            random_state=42
        )
        
        logger.info("Training spot model...")
        spot_model.fit(X_train, y_spot_train)
        spot_score = spot_model.score(X_test, y_spot_test)
        
        logger.info("Training perpetual model...")
        perp_model.fit(X_train, y_perp_train)
        perp_score = perp_model.score(X_test, y_perp_test)
        
        logger.info(
            f"Model training complete\n"
            f"- Spot R²: {spot_score:.3f}\n"
            f"- Perp R²: {perp_score:.3f}"
        )
        
        return spot_model, perp_model
        
    except Exception as e:
        logger.error(f"Model building failed: {str(e)}")
        if 'X' in locals():
            logger.debug(f"Feature matrix shape: {X.shape}")
            logger.debug(f"NaN counts:\n{X.isna().sum()}")
        raise

# 3. Secondary Objectives

## 3.1 Price Discovery Leadership Analysis


In [6]:
### ----------------------------
### 5. SECONDARY ANALYSES
### ----------------------------

def analyze_price_discovery(impact_results):
    """Determine which market more often leads price discovery"""
    spot_lead = impact_results[impact_results['type'] == 'spot_to_perp']
    perp_lead = impact_results[impact_results['type'] == 'perp_to_spot']
    
    spot_effectiveness = spot_lead['response_magnitude'].mean() / spot_lead['initiator_magnitude'].mean()
    perp_effectiveness = perp_lead['response_magnitude'].mean() / perp_lead['initiator_magnitude'].mean()
    
    return {
        'spot_lead_count': len(spot_lead),
        'perp_lead_count': len(perp_lead),
        'spot_effectiveness': spot_effectiveness,
        'perp_effectiveness': perp_effectiveness,
        'dominant_market': 'spot' if len(spot_lead) > len(perp_lead) else 'perpetual'
    }

## 3.2 Noise Event Detection


In [7]:
def detect_noise_events(jump_data, impact_results, window_ms=10):
    """Identify price jumps that don't propagate to the other market"""
    all_jumps = jump_data[(jump_data['spot_jump'] == 1) | (jump_data['perp_jump'] == 1)]
    noise_events = []
    
    for _, row in all_jumps.iterrows():
        # Check if this jump was analyzed in impact results
        matched = impact_results[
            (impact_results['timestamp'] == row['timestamp']) & 
            (
                (impact_results['type'] == 'spot_to_perp' and row['spot_jump'] == 1) |
                (impact_results['type'] == 'perp_to_spot' and row['perp_jump'] == 1)
            )
        ]
        
        # If no impact found, it's a potential noise event
        if matched.empty:
            future_window = jump_data[
                (jump_data['timestamp'] > row['timestamp']) & 
                (jump_data['timestamp'] <= row['timestamp'] + timedelta(milliseconds=window_ms))
            ]
            
            if not future_window.empty:
                # Check if price reverted
                if row['spot_jump'] == 1:
                    init_price = row['mid_spot']
                    future_price = future_window['mid_spot'].iloc[-1]
                    reverted = abs((future_price - init_price)/init_price) < 0.0002
                else:
                    init_price = row['mid_perp']
                    future_price = future_window['mid_perp'].iloc[-1]
                    reverted = abs((future_price - init_price)/init_price) < 0.0002
                
                if reverted:
                    noise_events.append({
                        'timestamp': row['timestamp'],
                        'market': 'spot' if row['spot_jump'] == 1 else 'perpetual',
                        'direction': row['spot_direction'] if row['spot_jump'] == 1 else row['perp_direction'],
                        'magnitude': row['spot_pct_change'] if row['spot_jump'] == 1 else row['perp_pct_change']
                    })
    
    return pd.DataFrame(noise_events)

## 3.3 Momentum Quality Estimation


In [8]:
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

def analyze_momentum(jump_data, impact_results, duration_ms=50):
    """Analyze how long price movements sustain their momentum"""
    momentum_results = []
    
    for _, event in impact_results.iterrows():
        # Get future price path
        future_prices = jump_data[
            (jump_data['timestamp'] > event['timestamp']) & 
            (jump_data['timestamp'] <= event['timestamp'] + timedelta(milliseconds=duration_ms))
        ]
        
        if not future_prices.empty:
            if event['type'] == 'spot_to_perp':
                price_series = future_prices['mid_spot']
                init_price = jump_data.loc[jump_data['timestamp'] == event['timestamp'], 'mid_spot'].iloc[0]
            else:
                price_series = future_prices['mid_perp']
                init_price = jump_data.loc[jump_data['timestamp'] == event['timestamp'], 'mid_perp'].iloc[0]
            
            # Calculate momentum metrics
            max_move = (price_series.max() - init_price)/init_price if event['initiator_direction'] > 0 else (init_price - price_series.min())/init_price
            sustain_period = len(price_series)
            half_life = None
            
            # Find when momentum drops below half of max move
            if event['initiator_direction'] > 0:
                threshold = init_price * (1 + max_move/2)
                cross_points = price_series[price_series < threshold]
            else:
                threshold = init_price * (1 - max_move/2)
                cross_points = price_series[price_series > threshold]
            
            if not cross_points.empty:
                half_life = (cross_points.index[0] - event['timestamp']).total_seconds() * 1000  # in ms
            
            momentum_results.append({
                'timestamp': event['timestamp'],
                'market': 'spot' if event['type'] == 'spot_to_perp' else 'perpetual',
                'direction': event['initiator_direction'],
                'max_move': max_move,
                'sustain_period': sustain_period,
                'half_life': half_life,
                'momentum_quality': max_move * (half_life if half_life else sustain_period)
            })
    
    return pd.DataFrame(momentum_results)


# 4. Integrated Analysis Pipeline


In [9]:
def full_analysis_pipeline(spot_path, perp_path, trade_path=None):
    """Complete pipeline with enhanced validation"""
    try:
        # 1. Load and preprocess
        print("1/6 Loading and preprocessing data...")
        spot_data = pd.read_csv(spot_path)
        perp_data = pd.read_csv(perp_path)
        trade_data = pd.read_csv(trade_path) if trade_path else None
        
        spot_data, perp_data, trade_data = preprocess_data(spot_data, perp_data, trade_data)
        
        # 2. Feature engineering
        print("2/6 Creating features...")
        features = create_features(spot_data, perp_data, trade_data)
        
        if len(features) < 10:  # Minimum data points required
            raise ValueError(f"Only {len(features)} valid rows after preprocessing - need more data")
        
        # 3. Price jump detection
        print("3/6 Detecting price jumps...")
        jump_data = detect_price_jumps(features, lookback_ms=3)
        
        # 4. Market impact analysis
        print("4/6 Analyzing market impact...")
        impact_results = analyze_market_impact(jump_data, window_ms=5)
        
        # 5. Model building with validation
        print("5/6 Building models...")
        try:
            spot_model, perp_model = build_predictive_models(features)
        except ValueError as e:
            print("\nModel building failed. Debugging info:")
            print(f"- Features shape: {features.shape}")
            print(f"- Numeric columns: {features.select_dtypes(include=[np.number]).columns.tolist()}")
            print(f"- NaN counts:\n{features.isna().sum()}")
            raise
        
        # 6. Secondary analyses
        print("6/6 Running secondary analyses...")
        discovery_results = analyze_price_discovery(impact_results)
        noise_events = detect_noise_events(jump_data, impact_results, window_ms=10)
        momentum_results = analyze_momentum(jump_data, impact_results, duration_ms=50)
        
        return {
            'processed_data': features,
            'models': {'spot': spot_model, 'perp': perp_model},
            'impact_results': impact_results,
            'discovery_results': discovery_results,
            'noise_events': noise_events,
            'momentum_results': momentum_results
        }

    except Exception as e:
        print(f"\nPipeline failed: {str(e)}")
        
        # Additional debugging info
        if 'features' in locals():
            print("\nFeature dataframe sample:")
            print(features.head(3))
            print("\nFeature columns:", features.columns.tolist())
            print("Numeric columns:", features.select_dtypes(include=[np.number]).columns.tolist())
        
        raise ValueError("Analysis failed - see above for details") from e

# 5. Debugging

## 5.1

In [ ]:
def inspect_data_files(spot_path, perp_path, trade_path=None):
    """Inspect the structure of input files"""
    print("=== Spot Data ===")
    spot_data = pd.read_csv(spot_path)
    print(f"Columns: {spot_data.columns.tolist()}")
    print(f"First row:\n{spot_data.iloc[0]}\n")
    
    print("=== Perp Data ===")
    perp_data = pd.read_csv(perp_path)
    print(f"Columns: {perp_data.columns.tolist()}")
    print(f"First row:\n{perp_data.iloc[0]}\n")
    
    if trade_path:
        print("=== Trade Data ===")
        trade_data = pd.read_csv(trade_path)
        print(f"Columns: {trade_data.columns.tolist()}")
        print(f"First row:\n{trade_data.iloc[0]}\n")

# Usage:
inspect_data_files(
    SPOT_PATH,
    PERP_PATH,
    TRADES_PATH
)

=== Spot Data ===
Columns: ['time', 'symbol', 'bid_price', 'ask_price']
First row:
time         2025-05-28 11:48:51.391000+00:00
symbol                               TRB_USDT
bid_price                               50.57
ask_price                               50.58
Name: 0, dtype: object

=== Perp Data ===
Columns: ['time', 'symbol', 'bid_price', 'ask_price']
First row:
time         2025-05-28 11:48:50.593+00
symbol                         TRB_USDT
bid_price                        50.537
ask_price                        50.538
Name: 0, dtype: object

=== Trade Data ===
Columns: ['time', 'symbol', 'price', 'quantity', 'is_market_maker']
First row:
time               2025-05-28 11:48:48.377+00
symbol                               TRB_USDT
price                                   50.56
quantity                                  0.2
is_market_maker                             t
Name: 0, dtype: object



In [ ]:
try:
    spot_data = pd.read_csv(spot_path)
    perp_data = pd.read_csv(perp_path)
    trade_data = pd.read_csv(trade_path) if trade_path else None
    
    logger.debug(f"Raw data loaded - spot: {len(spot_data)} rows, perp: {len(perp_data)} rows")
    
except Exception as e:
    logger.error(f"Error loading data files: {str(e)}")
    raise

# 2. Preprocess with our robust functions
logger.info("Preprocessing data...")
spot_data, perp_data, trade_data = preprocess_data(
    spot_data, 
    perp_data, 
    trade_data,
    handle_mm_nulls='fill_false'
)

2025-07-10 13:08:29 - market_analysis - ERROR - Error loading data files: name 'spot_path' is not defined


NameError: name 'spot_path' is not defined

## 5.2

In [26]:
try:
    full_analysis_pipeline(
    SPOT_PATH,
    PERP_PATH,
    TRADES_PATH
    )

except Exception as e:
    logger.error(f"Full analysis pipeline failed: {str(e)}")
    raise

1/6 Loading and preprocessing data...
2025-07-10 17:50:05 - market_analysis.preprocessor - INFO - Preprocessing complete
- Spot: 1153591 rows
- Perp: 5593817 rows
- Trade: 291966 rows
2/6 Creating features...
2025-07-10 17:50:05 - market_analysis.feature_engineering - INFO - Creating features...
2025-07-10 17:50:05 - market_analysis.feature_engineering - INFO - Validating input data...
2025-07-10 17:50:06 - market_analysis.feature_engineering - INFO - Merging spot and perpetual data...
2025-07-10 17:50:06 - market_analysis.feature_engineering - INFO - Merged data contains 1153591 initial rows
2025-07-10 17:50:06 - market_analysis.feature_engineering - INFO - Calculating market features...
2025-07-10 17:50:06 - market_analysis.feature_engineering - INFO - Processing trade data...
2025-07-10 17:50:07 - market_analysis.feature_engineering - INFO - Added 4 trade features
2025-07-10 17:50:07 - market_analysis.feature_engineering - INFO - Handling missing values with diagnostics...
2025-07-1

ValueError: Analysis failed - see above for details